<b>In Domain Query Template</b>
```python
STROKE_TEMPLATE = {
    "player_stroke_area": [
        "When does the {player} hits a {stroke} {hit_area}?",
        "At what point in the rally does the {player} perform a {stroke} {hit_area}?",
        "Identify when the {player} executes a {stroke} {hit_area}.",
        "Locate the moment when a {stroke} {hit_area} is played by the {player}."
    ],
    "player_stroke": [
        "When does the {player} hits a {stroke}?",
        "At what time does the {player} execute a {stroke}?",
        "When in this rally does the {player} perform a {stroke}?",
        "Identify the moment the {player} plays a {stroke}."
    ],
    "stroke_only": [
        "When is a {stroke} hits?",
        "At what moment does a {stroke} occur?",
        "When during the rally does a {stroke} happen?",
        "Locate when the {stroke} takes place."
    ],
    "hit_area_only": [
        "Which stroke is hit {hit_area}?",
        "Which shot lands {hit_area}?",
        "Identify the stroke that occurs {hit_area}.",
        "What stroke happens {hit_area}?"
    ]
}
STRATEGIES_TEMPLATE = {
    "four_corner": [
        "When does the players execute the four-corner pattern during a rally?",
        "Identify the timestamps where a corner-to-corner tactic occurs.",
        "At which points in the match do both backcourt corners get targeted in succession?"
    ],
    "net_shot": [
        "When do we see consecutive net-play exchanges in the rally?",
        "Locate moments of close-to-the-net shot sequences.",
        "At what times do players perform back-and-forth shots right at the net?"
    ],
    "back_court": [
        "When does a deep-court attack pattern appear?",
        "Pinpoint the instances of back-court rallies.",
        "At which moments are both rear-court zones hit consecutively?"
    ],
    "flat_shot_sequence": [
        "When do we observe a series of flat-drive strokes?",
        "Identify the moments of back-to-back flat-shot exchanges.",
        "At what points does a continuous flat-shot sequence unfold?"
    ],
    "upper_player_counter_attack": [
        "When does the upper player launch a counter-attack after a defensive shot?",
        "Identify moments where the upper player switches from defense to offense.",
        "At which timestamps does the upper player execute a defensive counterstrike?"
    ],
    "bottom_player_counter_attack": [
        "When does the bottom player initiate a counter-attack following a defensive return?",
        "Locate the points where the bottom player transitions from defense into an aggressive shot.",
        "At what moments does the bottom player perform a defensive riposte into attack?"
    ]
}
```

In [1]:
import re
import os
import warnings
from IPython.display import Video, display


warnings.filterwarnings("ignore", category=FutureWarning)

def extract_ans(text: str) -> tuple[list[int], str]:
    thinking_match = re.search(r"<thinking>(.*?)</thinking>", text, flags=re.DOTALL)
    thinking_str = thinking_match.group(1).strip() if thinking_match else ""

    m = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL)
    if m:
        nums = re.findall(r"\b\d+\b", m.group(1))
        return list(map(int, nums)), thinking_str

    m2 = re.search(r"The event happens at strokes? ([\d,]+)", text)
    if m2:
        nums = m2.group(1).split(",")
        return [int(n) for n in nums], thinking_str

    return [], thinking_str


## Init Model

In [2]:
from engines.instructblip_engine import InstructBLIPBadmintonVQAEngine  
engine = InstructBLIPBadmintonVQAEngine(
    cfg_path="lavis/projects/instructblip/inference/inference_instructblip_badminton_qa_coT_3.yaml",
    device="cuda",
    amp=True,
    clip_cache_size=0
)

/home/aiden/miniconda3/envs/PEFT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of trainable parameters in Qformer: 0
applying llm lora on all
trainable params: 6,488,064 || all params: 2,856,146,944 || trainable%: 0.22716142156585065


## Inference

In [3]:
video_root = "lavis/configs/datasets/badminton_caption/input/images"
clips = [
    "game1_set1_26511.mp4",
    "game1_set1_26531.mp4",
    "game1_set1_26558.mp4",
    "game1_set1_26570.mp4",
    "game1_set1_26591.mp4",
    "game1_set1_26612.mp4",
    "game1_set1_26645.mp4",
    "game1_set1_26662.mp4",
    "game1_set1_26683.mp4"
]
clip_paths = [os.path.join(video_root, clip) for clip in clips]
question = "When does the upper player hits a serve short?"

outputs, samples = engine.predict([clip_paths], question)

for output in outputs:
    ans, thinking = extract_ans(output)
    video_path = [clip_paths[i] for i in ans]
    print(f"Thinking: {thinking}")
    print(f"Video Paths: {video_path}")

    for vp in video_path:
        display(Video(vp, embed=True))

Thinking: I need to identify when the particular player performs a specific stroke. Therefore, we focus on player and stroke type. stroke 0: upper player hits a serve short stroke 1: bottom player hits a net pop stroke 2: upper player hits a drop net stroke 3: bottom player hits a cross-court net shot stroke 4: upper player hits a net lift stroke 5: bottom player hits a net pop stroke 6: upper player hits a drop net stroke 7: bottom player hits a net lift stroke 8: upper player hits a net lift therefore the answer is stroke 0
Video Paths: ['lavis/configs/datasets/badminton_caption/input/images/game1_set1_26511.mp4']
